In [ ]:
# ==========================================
# Cell 1: ライブラリのインポートと環境確認
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision.transforms as transforms
import urllib.request

# 1. GPUが使えるかどうかの確認
# Colabのメニュー「ランタイム」->「ランタイムのタイプを変更」でGPUを選択してください
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== 実行環境の確認 ===")
print(f"PyTorchのバージョン: {torch.__version__}")
print(f"使用しているデバイス: {device}")

if device.type == 'cuda':
    print(f"GPUのモデル: {torch.cuda.get_device_name(0)}")
    print("素晴らしい！GPUが正しく設定されています。計算が高速になります。")
else:
    print("注意: 現在CPUで実行しています。ディープラーニングにはGPUの利用を推奨します。")


In [ ]:
# ==========================================
# Cell 2: NumPyとPandasの基本
# ==========================================

print("=== 1. NumPyの基本（数値計算） ===")
# NumPyは配列（行列）の計算を高速に行うライブラリです。画像も本質的にはNumPyの配列として扱えます。
numpy_array = np.array([[1, 2, 3], [4, 5, 6]])
print("NumPy配列:\n", numpy_array)
print("配列の形状 (Shape):", numpy_array.shape) # (2, 3) -> 2行3列

print("\n=== 2. Pandasの基本（データ管理） ===")
# Pandasは表形式のデータを扱うライブラリです。機械学習ではデータセットの管理によく使われます。
# 例として、架空の画像データセットの情報を表にしてみます。
data = {
    '画像ファイル名': ['image_01.jpg', 'image_02.jpg', 'image_03.jpg'],
    '画像サイズ': ['256x256', '512x512', '256x256'],
    '正解ラベル': ['犬', '猫', '車']
}
df = pd.DataFrame(data)

# 表（データフレーム）を表示
display(df) 


In [ ]:
# ==========================================
# Cell 3: 画像の読み込み (PIL) と表示 (Matplotlib)
# ==========================================

# サンプル画像をインターネットからダウンロード（Colab環境用）
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/600px-Cat03.jpg"
image_path = "sample_cat.jpg"
urllib.request.urlretrieve(url, image_path)

# 1. PIL (Python Imaging Library) で画像を読み込む
print("=== 画像の読み込み ===")
img = Image.open(image_path)

# 画像の基本情報を出力
print(f"画像のフォーマット: {img.format}")
print(f"画像のサイズ (幅 x 高さ): {img.size}")
print(f"画像のカラーモード: {img.mode}") # RGBなど

# 2. Matplotlibで画像を表示
plt.figure(figsize=(6, 4))
plt.imshow(img)
plt.title("Loaded Image (PIL & Matplotlib)")
plt.axis('off') # 軸の目盛りを消す
plt.show()

# 3. 画像をNumPy配列に変換して、中身の数字を見てみる
img_numpy = np.array(img)
print(f"\n画像をNumPy配列に変換した後の形状: {img_numpy.shape}") 
# 結果は (高さ, 幅, チャンネル数(RGBの3)) になります。


In [ ]:
# ==========================================
# Cell 4: PyTorchのテンソル (Tensor) と画像変換
# ==========================================

print("=== 1. 基本的なテンソルの作成 ===")
# PyTorchのテンソルは、NumPyの配列と非常に似ていますが、GPUで計算できるのが最大の違いです。
tensor_a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("PyTorchテンソル:\n", tensor_a)

print("\n=== 2. PIL画像をPyTorchテンソルに変換 ===")
# torchvision.transformsを使って、画像をディープラーニング用のテンソルに変換します。
# ToTensor()は、0-255のピクセル値を0.0-1.0の範囲に自動で正規化してくれます。
transform = transforms.ToTensor()
img_tensor = transform(img)

print("変換前のNumPy形状 (H, W, C):", img_numpy.shape)
# PyTorchでは、チャンネル(C)が前に来るルールがあります。
print("変換後のTensor形状 (C, H, W):", img_tensor.shape) 

# ディープラーニングのモデルは「バッチサイズ」を要求するため、次元を1つ追加します (B, C, H, W)
img_tensor_batched = img_tensor.unsqueeze(0)
print("バッチ次元追加後の形状 (B, C, H, W):", img_tensor_batched.shape)

print("\n=== 3. テンソルをGPUへ転送 ===")
# モデルに入力する前に、データをGPUメモリに移動させる必要があります。
if torch.cuda.is_available():
    img_tensor_gpu = img_tensor_batched.to(device)
    print(f"データの現在の場所 (デバイス): {img_tensor_gpu.device}")
    print("これでモデルに入力する準備が整いました！")
else:
    print("GPUが利用できないため、データはCPUに残ります。")
